# Hessian computations with Dynamiqs

This notebook is a worked example for using the Hessian-compatible Dynamiqs solver path introduced for `dq.sesolve()` and `dq.mesolve()`.

The goal is practical: define a scalar-valued function from a normal Dynamiqs simulation, then call `jax.hessian(...)` on it. This is useful for curvature-aware optimization, uncertainty quantification, sensitivity analysis, and diagnosing which model parameters actually affect an observable.

We will use two basic examples:

1. A closed qubit solved with `dq.sesolve()`, where the Hessian is a scalar.
2. A lossy harmonic oscillator solved with `dq.mesolve()`, where the Hessian is a full matrix with respect to

$$
\theta = (\omega, \kappa, \alpha_0).
$$

The key solver option is:

```python
gradient=dq.gradient.Hessian()
```

Internally, this selects a higher-order-autodiff-friendly Diffrax configuration, while the public workflow remains standard JAX + Dynamiqs.


## Setup

Double precision is used for Hessian checks because second derivatives amplify solver and truncation errors. 


In [ ]:
import jax
import jax.numpy as jnp
import dynamiqs as dq

from dynamiqs.gradient import Hessian
from dynamiqs.method import Tsit5

# Hessian comparisons are much cleaner in double precision.
dq.set_precision('double'
)
print(f'Dynamiqs version: {dq.__version__}')
print(f'JAX x64 enabled: {jax.config.read("jax_enable_x64")}')


## Example 1 — closed qubit with `dq.sesolve()`

Consider a one-parameter Hamiltonian

$$
H(\omega)=\frac{\omega}{2}\sigma_x,
$$

with initial state

$$
|\psi_0\rangle=|0\rangle.
$$

For the final observable $\sigma_z$ at time $t$, the analytical expectation value is

$$
\langle \sigma_z \rangle(\omega)=\cos(\omega t),
$$

so

$$
\frac{d^2}{d\omega^2}\langle \sigma_z \rangle(\omega)
= -t^2\cos(\omega t).
$$

This is a compact smoke test for the full public `sesolve()` save path: the scalar function below returns `result.expects[0, -1].real`, then JAX computes the Hessian through the solve.


In [ ]:
sx = dq.sigmax()
sz = dq.sigmaz()
psi0 = dq.basis(2, 0)

omega = jnp.asarray(0.3)
t_final = jnp.asarray(0.7)
tsave = jnp.asarray([0.0, t_final])
method = Tsit5(rtol=1e-10, atol=1e-10)


def final_sigmaz_sesolve(omega):
    H = 0.5 * omega * sx
    result = dq.sesolve(
        H,
        psi0,
        tsave,
        exp_ops=[sz],
        method=method,
        gradient=Hessian(),
        progress_meter=False,
    )
    return result.expects[0, -1].real


hess_sesolve = jax.hessian(final_sigmaz_sesolve)(omega)
expected_sesolve = -(t_final**2) * jnp.cos(omega * t_final)

print(f'computed Hessian : {hess_sesolve:.12f}')
print(f'analytical value : {expected_sesolve:.12f}')
print(f'absolute error   : {jnp.abs(hess_sesolve - expected_sesolve):.3e}')


## Example 2 — open oscillator curvature with `dq.mesolve()`

Now we use a small open-system example inspired by the Dynamiqs basic gradient tutorial: a lossy harmonic oscillator with

$$
H = \omega a^\dagger a,
\qquad
L = \sqrt{\kappa}a,
\qquad
|\psi(0)\rangle = |\alpha_0\rangle.
$$

We measure the final photon number

$$
\bar n(T) = \mathrm{Tr}\left[a^\dagger a \rho(T)\right].
$$

For an initially coherent state evolving under this model,

$$
\alpha(T)=\alpha_0 e^{-\kappa T/2}e^{-i\omega T},
$$

and therefore

$$
\bar n(T)=|\alpha(T)|^2=\alpha_0^2 e^{-\kappa T}.
$$

That expression is independent of $\omega$: the frequency rotates the phase-space point but does not change its radius. The Hessian with respect to

$$
\theta = (\omega, \kappa, \alpha_0)
$$

is

$$
\nabla^2_\theta \bar n(T)=
\begin{pmatrix}
0 & 0 & 0 \\
0 & \alpha_0^2 T^2 e^{-\kappa T} & -2\alpha_0 T e^{-\kappa T} \\
0 & -2\alpha_0 T e^{-\kappa T} & 2e^{-\kappa T}
\end{pmatrix}.
$$

This matrix has a nice interpretation: $\omega$ has zero curvature for this observable, while $\kappa$ and $\alpha_0$ control the final population magnitude.


In [ ]:
n = 8
T = jnp.asarray(1.3)
tsave = jnp.linspace(0.0, T, 41)
method = Tsit5(rtol=1e-8, atol=1e-8)

# theta = (omega, kappa, alpha0)
theta = jnp.asarray([1.0, 0.2, 0.7])


def final_photon_number_mesolve(theta):
    omega, kappa, alpha0 = theta

    a = dq.destroy(n)
    H = omega * a.dag() @ a
    jump_ops = [jnp.sqrt(kappa) * a]
    psi0 = dq.coherent(n, alpha0)

    result = dq.mesolve(
        H,
        jump_ops,
        psi0,
        tsave,
        exp_ops=[dq.number(n)],
        method=method,
        gradient=Hessian(),
        progress_meter=False,
    )
    return result.expects[0, -1].real


def analytical_population_hessian(theta):
    _, kappa, alpha0 = theta
    decay = jnp.exp(-kappa * T)
    return jnp.asarray(
        [
            [0.0, 0.0, 0.0],
            [0.0, alpha0**2 * T**2 * decay, -2.0 * alpha0 * T * decay],
            [0.0, -2.0 * alpha0 * T * decay, 2.0 * decay],
        ]
    )


hess_mesolve = jax.hessian(final_photon_number_mesolve)(theta)
expected_mesolve = analytical_population_hessian(theta)

print('computed Hessian:')
print(hess_mesolve)
print('\nanalytical Hessian:')
print(expected_mesolve)
print('\nmax absolute error:', jnp.max(jnp.abs(hess_mesolve - expected_mesolve)))


## For Dynamiqs users: How to use this in your own model?

Some helpful steps:

1. Pick parameters $\theta$ and build all parameter-dependent objects inside a scalar Python function.
2. Call `dq.sesolve()` or `dq.mesolve()` with `gradient=Hessian()`.
3. Return a real scalar from a public saved output, for example, an expectation value of the final state.
4. Call `jax.hessian(f)(theta)`.

A typical template is:

```python
def objective(theta):
    H, jump_ops, y0, tsave = build_model(theta)
    result = dq.mesolve(
        H,
        jump_ops,
        y0,
        tsave,
        exp_ops=[observable],
        gradient=dq.gradient.Hessian(),
        progress_meter=False,
    )
    return result.expects[0, -1].real

curvature = jax.hessian(objective)(theta)
```

A practical note: if the Hessian is noisy, tighten solver tolerances, reduce Hilbert-space truncation error, and consider enabling double precision. Hessians are often more sensitive to numerical error than first derivatives.
